# Suite XDEF — C++ definitions, read from the store

The names a C++ translation unit declares, in the scopes C++ gives them (`Cpp.Definitions.define(store, path)`),
read from the unit the store holds and computed on demand, nothing stored: `Programs.C.Definitions`' definer over
C++'s kinds and terms, its global scope a namespace and its qualified names joined by `::`. One entity per name, kind
and signature per scope, merging a function's declarations with its out-of-line definition and a class's declaration
with its body; namespaces reopened, nested, inline, unnamed and aliased; classes, their members, bases and nested
types; templates' parameters in a scope of their own; using-directives, using-declarations and `using enum`; lambdas'
captures and parameters; statements that declare; labels, macros, modules and linkage specifications. A place is an
`Anchor`, plain data that outlives the transaction it was read in, written as text
(`body[1].body.body[0]:declarators[0].declarator.name`), by which each case names the places it checks; a qualified, a
template's or an operator's name is one name at its place (`ns::y`, `operator+`). Each case parses a unit into one
store (`Cpp.Parse.parse`), defines it, and checks the entities, their declarations, and what names refer to. The cases
port 0.3's C++ cases of DEF (`05_Definitions.ipynb`).

In [ ]:
import itertools
import time
from pathlib import Path

from mbse.Expressions.Framework import Objects, Terms
from mbse.Programs import Cpp
from mbse.Programs.Cpp import Definitions as D, Parse, Statements
from mbse.Programs.Framework.Model import WORKSPACE
from mbse.Schemas.Framework import Schemas
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Cpp.store()
PATHS = (f"case{n}.cpp" for n in itertools.count())


def define(text):
    """The unit `text` parses to, written in the store at a path of its own."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return D.define(STORE, path)


def written(*body):
    """The unit whose `body` is the statements `body`, each written in builder syntax as a branch of
    `Programs.Cpp.Statement`, as `Parse.parse` writes one, at a path of its own."""
    path = next(PATHS)

    def transaction(txn):
        return txn.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({"ws": txn.Programs.Framework.Workspace, "unit": txn.Programs.Cpp.TranslationUnit})
            .spec([
                lambda _: _.ws.files({path: lambda e: e.file(_.unit)}),
                lambda _: _.unit.body(list(body)),
            ])]))])

    STORE.transaction(transaction)
    return D.define(STORE, path)


def listing(program):
    return [(e.kind, e.qualified_name(), len(e.declarations)) for e in program.entities()]


def refers(program, at):
    return [f"{e.kind}:{e.qualified_name()}" for e in D.referents(program, at)]


def named(program, spelling):
    """The places a name spelled `spelling` is written, as text, in source order."""
    return [str(at) for at in program.named(spelling)]

## XDEF-01 · Namespaces: reopened, nested, inline, unnamed, aliased and used; using-declarations and `using enum`

In [ ]:
program = define("namespace a { int x; } namespace a { int y; } namespace a::b::inline c { int z; } "
                 "inline namespace v1 { int w; } namespace { int hidden; } namespace { int also; } "
                 "namespace n = a::b; namespace m = missing; namespace q = n; "
                 "namespace u { using namespace a; using namespace ::a; int k = x; } "
                 "using a::x; using none::y; enum class E { R, G }; void f() { using enum E; using enum E; R; } "
                 "namespace [[deprecated]] old {} using enum a;")
a = program.lookup("a")[0]
assert len(program.lookup("a")) == 1 and len(a.declarations) == 3 and a.scope.lookup("y")[0].kind == "variable"
assert a.declarations == ["body[0]:names[0].name", "body[1]:names[0].name", "body[2]:names[0].name"]
assert program.lookup("a::b::z")[0].qualified_name() == "a::b::c::z"  # through the inline namespace
assert program.lookup("w")[0].qualified_name() == "v1::w" and program.lookup("hidden")[0].kind == "variable"
unnamed = [e for e in program.entities() if e.kind == "namespace" and e.name is None]
assert len(unnamed) == 1 and unnamed[0].declarations == ["body[4]", "body[5]"] and program.lookup("also")
assert program.entity_of("body[4]") is program.entity_of("body[5]") is unnamed[0]
assert program.entity_of("body[2]") is program.lookup("a::b::c")[0]  # the innermost of its names
assert program.lookup("n")[0].target is program.lookup("a::b")[0] and program.lookup("m")[0].target is None
assert program.lookup("q::z")[0].name == "z" and program.lookup("n::z") == program.lookup("q::z")
assert program.lookup("u")[0].scope.using == [a.scope]
assert refers(program, "body[9].body[2]:declarators[0].initializer.value.name") == ["variable:a::x"]  # the directive
assert refers(program, "body[9].body[1]:name") == ["namespace:a"] and named(program, "::a") == ["body[9].body[1]:name"]
assert program.root.lookup("x") == program.lookup("a::x")  # the using-declaration
assert program.entity_of("body[10]:declarators[0]") is program.lookup("a::x")[0]
using = next(e for e in program.entities() if e.kind == "using declaration")
assert using.name == "y" and using.parent is program.root and using.declarations == ["body[11]:declarators[0].name"]
assert refers(program, "body[13].body.body[2]:expression.name") == ["enumerator:E::R"]  # through `using enum`
assert program.lookup("R") == [] and refers(program, "body[15]:type") == ["namespace:a"]  # `using enum a;`: no enum
assert program.root.separator == "::" and program.root.kind == "namespace"
print(len(listing(program)), "entities")

## XDEF-02 · Classes: members in the class's scope, bases, out-of-line definitions, anonymous unions, nested classes, and friends declaring nothing

In [ ]:
program = define("""
struct B { int b; void g(); };
struct S;
struct S : B, missing::M {
    int field;
    static int count;
    static inline int ready = 1;
    static constexpr int size = 2;
    union { int i; float f; };
    struct { int depth; } info;
    void f() const;
    void f(int);
    friend class F;
    friend void g(S);
    friend void h() {}
};
int S::count = 0;
void S::f() const { field = b + size; }
void B::g() {}
struct A::Unknown { int u; };
typedef B Alias;
struct D : Alias { template <class T> friend void t(T); };
int S::extra = 3;
struct O { struct In; };
struct O::In { int z; };
""")
s = program.lookup("S")[0]
assert s.declarations == ["body[1]:specifiers[0].name", "body[2]:specifiers[0].name"]
assert s.definition == "body[2]:specifiers[0].name" and program.entity_of("body[2]:specifiers[0]") is s
assert [(e.kind, e.name) for e in s.scope.entities()] == [
    ("field", "field"), ("variable", "count"), ("variable", "ready"), ("variable", "size"), ("field", "info"),
    ("function", "f"), ("function", "f"), ("field", "extra")]  # unnamed classes are not found by name
count = program.lookup("S::count")[0]
assert count.declarations == ["body[2]:specifiers[0].members.members[1]:declarators[0].declarator.name",
                              "body[3]:declarators[0].declarator.name"]
assert count.definition == count.declarations[1]  # a static member's declaration in its class defines nothing
assert program.lookup("S::ready")[0].definition is not None and program.lookup("S::size")[0].definition is not None
assert program.lookup("S::i")[0].kind == "field"  # an anonymous union's members are its class's
assert program.lookup("S::depth") == []
const_f = next(e for e in program.lookup("S::f") if e.signature == "() const")
assert const_f.definition == "body[4]:declarator.declarator.name" and len(const_f.declarations) == 2
assert program.entity_of("body[4]") is const_f and const_f.scope.parent is s.scope  # nested in its class's
assert [e.signature for e in program.lookup("S::f") if e.kind == "function"] == ["() const", "(int)"]
assert program.lookup("B::g")[0].definition == "body[5]:declarator.declarator.name"
assert s.scope.bases == [program.lookup("B")[0].scope] and program.lookup("S::b")[0].parent.owner.name == "B"
assert refers(program, "body[4].body.body[0]:expression.left.name") == ["field:S::field"]  # from the member's body
assert refers(program, "body[4].body.body[0]:expression.right.left.name") == ["field:B::b"]  # through its base
assert program.lookup("F") == [] and program.lookup("h") == [] and program.lookup("S::g") == program.lookup("B::g")

assert refers(program, "body[2]:specifiers[0].members.members[8]:specifiers[1].name") == []  # `friend class F`
assert program.lookup("Unknown")[0].scope.lookup("u")[0].kind == "field"  # an unknown scope: declared where it is
assert program.lookup("D")[0].scope.bases == [] and program.lookup("t") == []  # an alias is no class here
assert program.lookup("S::extra")[0].parent is s.scope  # out of line, though the class does not declare it
assert program.lookup("O::In::z")[0].kind == "field" and len(program.lookup("O::In")[0].declarations) == 2
assert program.lookup("O::In")[0].scope.parent is program.lookup("O")[0].scope
print(len(listing(program)), "entities")

## XDEF-03 · Enumerations, aliases, concepts, templates and their parameters, specializations, and macros

In [ ]:
program = define("""
#define LIMIT 10
#define LIMIT 11
enum Color { Red, Green = LIMIT, };
enum class Mode : int { Off,
#ifdef X
  On,
#endif
};
enum Fwd : int;
typedef int Int, *IntPointer;
using Real = double;
template <typename T> concept Small = sizeof(T) < 4;
template <class T, int N, template <class> class W, typename... Ts>
struct Box { T items[N]; };
template <class, int> struct Unnamed;
template <class T> struct Box<T *> {};
template <class T> T identity(T t) { return t; }
template <class T> using Ptr = T *;
enum { Loose };
template int identity<int>(int);
enum Twice { One }; enum Twice { Two };
#ifdef LIMIT
#endif
""")
assert [e.kind for e in program.macros.entities()] == ["macro"] and len(program.macros.names["LIMIT"][0].declarations) == 2
assert refers(program, "body[2]:specifiers[0].enumerators[1].value.name") == ["macro:LIMIT"]
assert program.lookup("Red")[0].qualified_name() == "Color::Red" and program.lookup("Color::Red") == program.lookup(
    "Red")  # unscoped: its enclosing scope's, and its own
assert program.lookup("Off") == [] and program.lookup("Mode::On")[0].kind == "enumerator"  # scoped, a group's too
assert program.lookup("Fwd")[0].definition is None  # `enum Fwd : int;` declares it, without its enumerators
assert [e.kind for e in program.lookup("Int") + program.lookup("IntPointer") + program.lookup("Real")] == [
    "type alias"] * 3
assert program.lookup("Small")[0].kind == "concept" and program.entity_of("body[7]") is program.lookup("Small")[0]
box = program.lookup("Box")[0]
assert box.declarations == ["body[8].declaration:specifiers[0].name", "body[10].declaration:specifiers[0].name"]
assert box.definition == box.declarations[0] and program.entity_of("body[8]") is box  # a template declares its own
parameters = box.scope.parent
assert parameters.kind == "template" and [e.name for e in parameters.entities()] == ["T", "N", "W", "Ts"]
assert parameters.node == "body[8]" and box.parent is program.root  # its name, where the template stands
assert refers(program, "body[8].declaration:specifiers[0].members.members[0]:specifiers[0].name") == [
    "template parameter:T"]  # from its members
assert program.entity_of("body[10]") is box  # a partial specialization: a declaration of its primary template
assert refers(program, "body[10].declaration:specifiers[0].name.arguments[0].specifiers[0].name") == [
    "template parameter:T"]  # its arguments, the specialization's own parameters
assert program.entity_of("body[11]") is program.lookup("identity")[0]
assert program.entity_of("body[12]") is program.lookup("Ptr")[0]
assert program.scope_of("body[9]:parameters[0]").kind == "template" and program.lookup("Unnamed")[0].definition is None
assert program.lookup("Loose")[0].parent.owner.name is None  # an unnamed enumeration's, transparent to the unit
assert program.entity_of("body[14]") is None and refers(program, "body[14].declaration:declarators[0].declarator."
                                                         "declarator.name") == ["function:identity"]  # a use
twice = program.lookup("Twice")[0]  # defined twice, which C++ rejects: one entity, its scope its first body's
assert len(twice.declarations) == 2 and [e.name for e in twice.scope.entities()] == ["One"]
assert program.lookup("Two")[0].qualified_name() == "Twice::Two"
assert program.space_of("body[17]:directive.name") == "macro" and refers(program, "body[17]:directive.name") == [
    "macro:LIMIT"]
print(len(listing(program)), "entities")

## XDEF-04 · Functions: overloads by signature, parameters, blocks, statements that declare, labels, linkage, lambdas, `requires` and handlers

In [ ]:
program = define("""
void f(int);
void f(int x) { }
void f(double, ...);
void ::f(double, ...) {}
int (*pointer)(int);
extern int declared;
int declared = 1;
void body(int a, int) {
    int a2 = a;
    { int inner = a2; }
    if (int c = a; c) { int d = c; }
    for (int i = 0; i < a; ++i) {}
    for (int k : map) {}
    extern int later;
    int helper(int);
start:
    try { } catch (const E &e) { use(e); } catch (...) {}
    auto l = [a, n = 1]<typename T, int K, class>(T t, int) { return t + n + a; again: goto again; };
    auto r = requires (int q) { q; };
    goto start;
}
int later;
""")
f = program.lookup("f")
assert [(e.signature, len(e.declarations), e.definition is not None) for e in f] == [
    ("(int)", 2, True), ("(double, ...)", 2, True)]  # `::f`, the unit's

assert program.lookup("pointer")[0].kind == "variable"  # a pointer to a function is an object
declared = program.lookup("declared")[0]
assert len(declared.declarations) == 2 and declared.definition == declared.declarations[1]
body = program.lookup("body")[0]
assert body.scope.kind == "function" and body.scope.node == "body[7]" and body.scope.owner is body
assert [(e.kind, e.name) for e in body.scope.entities()] == [
    ("parameter", "a"), ("variable", "a2"), ("variable", "later"), ("function", "helper"), ("label", "start"),
    ("variable", "l"), ("variable", "r")]  # the body's outermost block is the function's scope, as C's
kinds = {}
for name in ("inner", "c", "d", "i", "k", "e", "n", "T", "K", "t", "q", "again"):
    found = [e for e in program.entities() if e.name == name]
    kinds[name] = (found[0].kind, found[0].parent.kind)
assert kinds == {"inner": ("variable", "block"), "c": ("variable", "block"), "d": ("variable", "block"),
                 "i": ("variable", "block"), "k": ("variable", "block"), "e": ("variable", "block"),
                 "n": ("variable", "lambda"), "T": ("template parameter", "lambda"),
                 "K": ("template parameter", "lambda"), "t": ("parameter", "lambda"), "q": ("parameter", "requires"),
                 "again": ("label", "lambda")}  # a lambda's labels its own
assert named(program, "a")[3:] == ["body[7].body.body[3]:condition.right.name",
                                   "body[7].body.body[8]:declarators[0].initializer.value.captures[0].name",
                                   "body[7].body.body[8]:declarators[0].initializer.value.block.body[0]:value.right.name"]
assert [refers(program, at) for at in named(program, "a")] == [["parameter:body::a"]] * 6  # a capture, a use
assert program.scope_of(named(program, "a")[4]) is body.scope  # a simple capture, where the lambda stands
later = program.lookup("later")[0]  # a block's `extern` declaration: the unit's entity, found from the block
assert later.parent is program.root and later.definition == "body[8]:declarators[0].declarator.name"
assert program.lookup("helper")[0].parent is program.root and program.lookup("helper")[0].definition is None
assert refers(program, "body[7].body.body[10]:label") == ["label:body::start"]
assert program.space_of("body[7].body.body[10]:label") == "label"
# A structured binding (C++17) binds each of its names, each name's place declaring its entity
program = define("auto [p, q] = pair;\nvoid g() {\n    for (auto &[k, v] : m) { use(k, v); }\n}\n")
assert listing(program) == [("variable", "p", 1), ("variable", "q", 1), ("function", "g", 1), ("variable", "k", 1),
                            ("variable", "v", 1)]
assert named(program, "p") == ["body[0]:declarators[0].declarator.bindings[0]"]
assert [refers(program, at) for at in named(program, "k")] == [["variable:k"]] * 2  # bound in the `for`'s block, used
assert program.scope_of(named(program, "k")[0]).kind == "block"
print(len(listing(program)), "entities")

## XDEF-05 · Names resolve from where they are, qualified names qualified, but members after `.` and `->` do not; a qualified, a template's, an operator's or a destructor's name is one name at its place

In [ ]:
program = define("""
int x;
namespace ns { int y; }
struct P { int x; template <class T> T get(); ~P(); int operator+(int); };
void f(P *p, P q) {
    int z = p->x + x + q.template get<P>() + ns::y + ::x + p->P::x + decltype(q)::x;
    p->~P();
}
""")
start = "body[3].body.body[0]:declarators[0].initializer.value"
assert refers(program, f"{start}{'.left' * 5}.right.name") == ["variable:x"]
assert program.anchor(f"{start}{'.left' * 6}.object") is not None
assert program.anchor(f"{start}{'.left' * 6}.member") is None  # `p->x`: a member, not visited
assert named(program, "x") == ["body[0]:declarators[0].declarator.name",
                               "body[2]:specifiers[0].members.members[0]:declarators[0].declarator.name",
                               f"{start}{'.left' * 5}.right.name"]
assert program.anchor(f"{start}{'.left' * 4}.right.function.member") is None  # `q.template get<P>()`
assert named(program, "ns::y") == [f"{start}{'.left' * 3}.right.name"]
assert refers(program, named(program, "ns::y")[0]) == ["variable:ns::y"]  # qualified
assert refers(program, named(program, "::x")[0]) == ["variable:x"]  # from the unit's scope
assert program.anchor(f"{start}.left.right.member") is None  # `p->P::x`
assert program.name_at(f"{start}.right.name") is None  # `decltype(q)::x`: no text names its qualifier
assert refers(program, f"{start}.right.name.qualifiers[0].expression.name") == ["parameter:f::q"]
assert program.anchor("body[3].body.body[1]:expression.function.member") is None  # `p->~P()`
assert named(program, "~P") == ["body[2]:specifiers[0].members.members[2]:declarators[0].declarator.declarator.name"]
assert refers(program, named(program, "P")[1]) == ["class:P"]  # `~P`'s class, a use
assert named(program, "operator+") == [
    "body[2]:specifiers[0].members.members[3]:declarators[0].declarator.declarator.name"]
assert program.lookup("P::operator+")[0].signature == "(int)" and program.lookup("P::~P")[0].kind == "function"
assert program.lookup("P::get")[0].parent is program.lookup("P")[0].scope  # a member template's, its class's
print("ok")

## XDEF-06 · Modules and linkage: an `export`ed declaration and a linkage specification's bind where they stand; a module's and an import's names name nothing

In [ ]:
program = define("""module;
export module geo.shapes:core;
import std;
export int area(int side);
export { struct Square { int side; }; }
export namespace geo { int origin; }
extern "C" int puts(const char *);
extern "C" { int printf(const char *, ...); }
int area(int side) { return side * side; }
module :private;
""")
assert listing(program) == [
    ("function", "area", 2), ("parameter", "side", 1), ("class", "Square", 1), ("field", "Square::side", 1),
    ("namespace", "geo", 1), ("variable", "geo::origin", 1), ("function", "puts", 1), ("function", "printf", 1),
    ("parameter", "area::side", 1)]
area = program.lookup("area")[0]
assert area.declarations == ["body[3].body[0]:declarators[0].declarator.declarator.name",
                             "body[8]:declarator.declarator.name"] and area.definition == area.declarations[1]
assert program.scope_of("body[4].body[0]") is program.root and program.scope_of("body[7].body[0]") is program.root
assert named(program, "geo.shapes") == [] and named(program, "std") == []  # strings, not names
assert program.anchor("body[1]") is not None and program.anchor("body[1]:name") is None  # a string, as written

print("ok")

## XDEF-07 · Every entity of the conformance sources, every name located, and what each declarator binds

Read independently of `define`, by a walk of the stored unit's value, as CDEF-07 reads C's: every statement,
declaration, clause, member and error, and every part holding them (a member list, a group, a block a lambda holds), is
located, its anchor's text its place in the unit's value, in source order. Every other place is located, unless it is
where a name names nothing or within a name: a member's name, a designator's, a macro's parameters, and the parts of a
qualified or a template's name, which is one name at its place; the pairs of kind and property where the walk stops are
listed. Every identifier, qualified name and template's name located has its name written there, and each declarator's
name, but in a type-id and a friend's, declares an entity of its name.

In [ ]:
HOLDERS = {f"{Statements.LANGUAGE}.{name}"
           for name in (*Statements.KINDS, "Error", "MemberList", "Group", "MemberGroup")}
LISTS = {f"{Statements.LANGUAGE}.{name}" for name in ("Body", "Members", "Handlers")}
NAMES = ("identifier", "qualifiedname", "templateid")


def record_of(value):
    """The record a value holds through unions, and its kind: a term's, or its schema's name in lower case."""
    while not Terms.is_term(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        ((_, value),) = Objects.properties(value).items()
    return Terms.kind_of(value) if Terms.is_term(value) else (value.as_schema().name.rsplit(".", 1)[-1].lower(), value)


def holds(record, name, value):
    """Whether a property holds statement places: statements, members, errors, a body, a member list or a group."""
    if record.as_schema().properties[name].schema.name in LISTS:
        return True
    return hasattr(value, "as_schema") and not Terms.is_term(value) and record_of(value)[1].as_schema().name in HOLDERS


def conformance(source):
    """The program of a conformance source, every place of it checked; the walk's counts, and where it stops."""
    STORE.transaction(Parse.parse((SOURCES / source).read_text(encoding="utf-8"), source))
    started = time.perf_counter()
    program = D.define(STORE, source)
    elapsed = time.perf_counter() - started
    anchors, unlocated, counts, friends = [], set(), {"names": 0, "declarators": 0}, []

    def walk(record, path):
        anchor = program.anchor(path or "<unit>")
        assert program.scope_of(anchor) is not None and anchor.property is None and anchor.statement == path, path
        anchors.append(anchor)
        for name, value in Objects.properties(record).items():
            if not holds(record, name, value):
                parts(value, f"{path}:{name}", (record_of(record)[0], name), False)
                continue
            for i, element in enumerate(value.values) if isinstance(value, Objects.Listed) else [(None, value)]:
                step = name if i is None else f"{name}[{i}]"
                walk(record_of(element)[1], f"{path}.{step}" if path else step)

    def parts(value, place, context, typed):
        if isinstance(value, Objects.Listed):
            for i, element in enumerate(value.values):
                parts(element, f"{place}[{i}]", context, typed)
            return
        if not hasattr(value, "as_schema"):
            return
        tag, record = record_of(value)
        if record.as_schema().name in HOLDERS:  # a block, a member list or a group a term holds
            walk(record, place)
            return
        if program.scope_of(place) is None:
            unlocated.add(context)
            return
        fields = Objects.properties(record)
        if tag in NAMES and program.name_at(place) is not None:
            counts["names"] += 1
            assert tag != "identifier" or program.name_at(place) == fields["spelling"], place
        if tag == "iddeclarator" and not typed:
            counts["declarators"] += 1
            entity = program.entity_of(f"{place}.name")
            if entity is None:
                friends.append(place)
            else:
                assert entity.name == program.name_at(f"{place}.name").rsplit("::", 1)[-1], place
        for name, held in fields.items():
            parts(held, f"{place}.{name}", (tag, name), typed or tag == "typeid")

    with STORE.transaction():
        walk(STORE.singleton(WORKSPACE).files({"path": source}), "")
    assert program.anchor("<unit>") == program.root.node == "<unit>"
    assert [a.order for a in anchors] == sorted(a.order for a in anchors)  # the walk's order is source order
    return program, len(anchors), counts, unlocated, friends, elapsed


kinds = {}
for source in ("cpp20.cpp", "cpp23.cpp"):
    program, statements, counts, unlocated, friends, elapsed = conformance(source)
    for entity in [*program.entities(), *program.macros.entities()]:
        kinds[entity.kind] = kinds.get(entity.kind, 0) + 1
    print(source, statements, "statement places,", counts, f"defined in {elapsed:.2f} s")
    if source == "cpp20.cpp":
        assert (statements, counts) == (247, {"names": 346, "declarators": 114}), (statements, counts)
        assert unlocated == {("memberexpression", "member"), ("fielddesignator", "name"),
                             ("definedirective", "parameters"), ("qualifiedname", "name"),
                             ("qualifiedname", "qualifiers"), ("templateid", "name")}, unlocated
        assert friends == ["body[35].declaration:specifiers[0].members.members[14]:declarators[0].declarator.declarator"]
        assert program.lookup("geo::v1::Point::x")[0].kind == "field"
        assert program.lookup("geo::Point") == program.lookup("geo::v1::Point")  # v1 is inline
        assert [e.signature for e in program.lookup("Vector::operator[]")] == ["(int)", "(int) const"]
        assert program.lookup("Vector::convert")[0].definition == "body[39].declaration:declarator.declarator.name"
        assert program.lookup("main::done")[0].kind == "label"
assert set(kinds) == set(D.MEANINGS), set(D.MEANINGS) - set(kinds)
print(sum(kinds.values()), "entities")

## XDEF-08 · Anchors, and the transaction `define` reads in

In [ ]:
program = define("""
int f(int a) {
    return a;
}
auto g = [](int b) { return a + b; };
""")
use = program.named("a")[1]
assert (use.statement, use.property, use.term) == ("body[0].body.body[0]", "value", ("name",)) and str(use) == (
    "body[0].body.body[0]:value.name")
assert repr(use) == "Anchor('body[0].body.body[0]:value.name')" and use == program.anchor(str(use))
within = program.named("a")[2]  # within a lambda's block
assert within.statement == "body[1]:declarators[0].initializer.value.block.body[0]"
with STORE.transaction():
    file = STORE.singleton(WORKSPACE).files({"path": program.path})
    joined = D.define_file(STORE, file)  # within the caller's transaction, from the unit's visitor
    with raises(ValueError, match="transactions do not nest"):  # define opens its own: called with none open
        D.define(STORE, program.path)
assert listing(joined) == listing(program) == [("function", "f", 1), ("parameter", "f::a", 1), ("variable", "g", 1),
                                               ("parameter", "b", 1)]
assert D.referents(joined, use) == [joined.lookup("f::a")[0]] and refers(program, within) == []
with raises(LookupError, match="absent.cpp"):
    D.define(STORE, "absent.cpp")
assert D.referents(program, "body[9]:value") == [] and program.anchor("body[9]:value") is None
assert program.lookup("::f") == program.lookup("f") and program.lookup("f::a", at=use) == program.lookup("f::a")
print("ok")

## XDEF-09 · Invalid syntax: an `Error` binds nothing itself, the statements recovered within it bind in its scope, wherever it stands; an `error` term's recovered terms are uses, and a `missing` term binds nothing

In [ ]:
program = define("""int a = 1;
void f() {
    a = 1;
    if (a {}
    int inner = 3;
    x = = 1;
}
struct s { int = ; int ok; };
int g = (a + );
""")
assert listing(program) == [("variable", "a", 1), ("function", "f", 1), ("variable", "f::inner", 1), ("class", "s", 1),
                            ("field", "s::ok", 1), ("variable", "g", 1)], listing(program)
assert program.anchor("body[1].body.body[1]") is not None and named(program, "x") == []  # its own text binds nothing
assert program.lookup("f::inner")[0].definition == "body[1].body.body[1].body[0]:declarators[0].declarator.name"
member = program.anchor("body[2]:specifiers[0].members.members[0]")  # an `Error` where a member stands
assert member is not None and program.scope_of(member) is program.lookup("s")[0].scope
assert named(program, "a")[-1] == "body[3]:declarators[0].initializer.value.expression.left.name"
assert program.anchor("body[3]:declarators[0].initializer.value.expression.right") is not None  # `missing`: no name
assert program.name_at("body[3]:declarators[0].initializer.value.expression.right") is None
# What C++ rejects but the schemas hold, written: a class with neither a name nor a body declares nothing, and a
# declarator's name no text names, `decltype(x)::y`, binds nothing
NAME = lambda n: n.identifier(lambda i: i.spelling("y"))  # noqa: E731
program = written(
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key("struct"))])),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.primitivetypespecifier(
        lambda k: k.keyword("int"))]).declarators([lambda d: d.declarator(lambda k: k.iddeclarator(lambda i: i.name(
            lambda n: n.qualifiedname(lambda q: q.qualifiers([lambda x: x.decltypespecifier(lambda t: t.expression(
                lambda e: e.idexpression(lambda i: i.name(lambda n: n.identifier(lambda x: x.spelling("x"))))))]).name(
                    NAME)))))])))
assert listing(program) == [] and program.scope_of("body[0]:specifiers[0]") is program.root
assert named(program, "y") == [] and named(program, "x") == [
    "body[1]:declarators[0].declarator.name.qualifiers[0].expression.name"]  # the decltype's, a use
print("ok")